# cjm-fasthtml-workflow-transcript-decomp

> A FastHTML-based human-in-the-loop workflow for decomposing raw transcripts into traversable context graph spines with precise text segmentation and audio alignment.

## Install


```bash
pip install cjm_fasthtml_workflow_transcript_decomp
```

## Project Structure

```
nbs/
├── core/ (1)
│   └── config.ipynb  # Configuration dataclass for structure decomposition workflow
├── routes/ (5)
│   ├── core/ (4)
│   │   ├── audio.ipynb    # Audio file serving route handlers for alignment playback
│   │   ├── init.ipynb     # Router assembly for core workflow routes
│   │   ├── sources.ipynb  # Source data retrieval route handlers
│   │   └── status.ipynb   # Workflow status and lifecycle route handlers
│   └── init.ipynb  # Router initialization for the structure decomposition workflow
└── workflow/ (1)
    └── workflow.ipynb  # Main workflow class for structure decomposition
```

Total: 7 notebooks across 3 directories

## Module Dependencies

```mermaid
graph LR
    core_config[core.config<br/>config]
    routes_core_audio[routes.core.audio<br/>audio]
    routes_core_init[routes.core.init<br/>init]
    routes_core_sources[routes.core.sources<br/>sources]
    routes_core_status[routes.core.status<br/>status]
    routes_init[routes.init<br/>init]
    workflow_workflow[workflow.workflow<br/>workflow]

    routes_core_audio --> workflow_workflow
    routes_core_init --> routes_core_status
    routes_core_init --> routes_core_audio
    routes_core_init --> routes_core_sources
    routes_core_init --> workflow_workflow
    routes_core_sources --> workflow_workflow
    routes_core_status --> workflow_workflow
    routes_init --> routes_core_init
    routes_init --> workflow_workflow
    workflow_workflow --> core_config
```

*10 cross-module dependencies detected*

## CLI Reference

No CLI commands found in this project.

## Module Overview

Detailed documentation for each module in the project:

### audio (`audio.ipynb`)
> Audio file serving route handlers for alignment playback

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.routes.core.audio import (
    DEBUG_AUDIO,
    init_audio_router
)
```

#### Functions

```python
def _handle_audio_src(
    workflow:StructureDecompWorkflow,  # The workflow instance
    sess,  # FastHTML session object
    path:str=None,  # Audio file path (from query parameter)
):  # Audio file response or 404
    "Serve an audio file by path for Web Audio API playback."
```

```python
def init_audio_router(
    workflow: StructureDecompWorkflow,  # The workflow instance
    prefix: str,  # Route prefix (e.g., "/workflow/core/audio")
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    "Initialize audio serving routes."
```

#### Variables

```python
DEBUG_AUDIO = False
```

### config (`config.ipynb`)
> Configuration dataclass for structure decomposition workflow

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.core.config import (
    DEFAULT_WORKFLOW_CONFIG_DIR,
    StructureDecompWorkflowConfig
)
```
#### Classes

```python
@dataclass
class StructureDecompWorkflowConfig:
    "Configuration for structure decomposition workflow."
    
    workflow_id: str = 'structure_decomposition'  # Unique identifier for this workflow
    route_prefix: str = '/structure_decomp'  # Base URL prefix for workflow routes
    stepflow_prefix: str = '/flow'  # Sub-prefix for StepFlow routes
    container_id: str = 'sd-workflow-container'  # HTML ID for main workflow container
    show_progress: bool = True  # Show step progress indicator in StepFlow
    max_history_depth: int = 500  # Maximum undo history entries for decomposition phase
    text_plugin: str = 'cjm-text-plugin-nltk'  # Text processing plugin for sentence splitting
    vad_plugin: str = 'cjm-media-plugin-silero-vad'  # VAD plugin for audio alignment
    graph_plugin: str = 'cjm-graph-plugin-sqlite'  # Graph plugin for storage
    fa_plugin_name: str = 'cjm-transcription-plugin-qwen3-forced-aligner'  # Forced alignment plugin (optional)
    source_categories: List[str] = field(...)  # Categories for source plugins
    no_plugins_redirect: Optional[str]  # URL to redirect when required plugins unavailable
    state_db_path: Optional[Path]  # Path to SQLite state database (uses default if None)
    config_dir: Path = field(...)  # Directory for workflow settings
    
    def get_full_stepflow_prefix(self) -> str:  # Combined route_prefix + stepflow_prefix
            """Get the full prefix for the StepFlow router."""
            return f"{self.route_prefix}{self.stepflow_prefix}"
        
        def get_state_db_path(self) -> Path:  # Resolved path to state database
        "Get the full prefix for the StepFlow router."
    
    def get_state_db_path(self) -> Path:  # Resolved path to state database
            """Get the path to the SQLite state database."""
            if self.state_db_path
        "Get the path to the SQLite state database."
```

#### Variables

```python
DEFAULT_WORKFLOW_CONFIG_DIR
```

### init (`init.ipynb`)
> Router assembly for core workflow routes

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.routes.core.init import (
    init_core_routers
)
```

#### Functions

```python
def init_core_routers(
    workflow: StructureDecompWorkflow,  # The workflow instance
    prefix: str,  # Base prefix for core routes (e.g., "/workflow/core")
) -> Tuple[List[APIRouter], Dict[str, Callable]]:  # (routers, merged_routes)
    """
    Initialize and return all core workflow routers.
    
    Note: Chrome switching route is now in cjm-transcript-segment-align library.
    It is initialized in routes/init.py alongside the other segment-align routes.
    """
```


### init (`init.ipynb`)
> Router initialization for the structure decomposition workflow

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.routes.init import (
    init_routers
)
```

#### Functions

```python
def init_routers(
    workflow: "StructureDecompWorkflow",  # The workflow instance
) -> List[APIRouter]:  # List of configured routers
    "Initialize and return all workflow routers."
```


### sources (`sources.ipynb`)
> Source data retrieval route handlers

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.routes.core.sources import (
    init_sources_router
)
```

#### Functions

```python
def _handle_get_sources(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    provider_id: str = None,  # Optional plugin name filter
    limit: int = 50  # Maximum number of results
):  # JSON response with transcription sources
    "Get available transcription sources."
```

```python
def init_sources_router(
    workflow: StructureDecompWorkflow,  # The workflow instance
    prefix: str,  # Route prefix (e.g., "/workflow/core/sources")
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    "Initialize source data routes."
```


### status (`status.ipynb`)
> Workflow status and lifecycle route handlers

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.routes.core.status import (
    init_status_router
)
```

#### Functions

```python
async def _handle_current_status(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess  # FastHTML session object
):  # Appropriate UI component based on current state
    "Return current workflow status - determines what to show."
```

```python
async def _handle_reset(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess  # FastHTML session object
):  # StepFlow start view
    "Reset workflow and return to start."
```

```python
def init_status_router(
    workflow: StructureDecompWorkflow,  # The workflow instance
    prefix: str,  # Route prefix (e.g., "/workflow/core/status")
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    "Initialize status and lifecycle routes."
```


### workflow (`workflow.ipynb`)
> Main workflow class for structure decomposition

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import (
    StructureDecompWorkflow
)
```

#### Functions

```python
@patch
def setup(
    self: StructureDecompWorkflow,
    app  # FastHTML application instance
) -> None
    "Initialize workflow with FastHTML app."
```

```python
@patch
def cleanup(
    self: StructureDecompWorkflow
) -> None
    "Clean up workflow resources."
```

```python
@patch
def get_routers(
    self: StructureDecompWorkflow
) -> List[APIRouter]:  # List of routers to register
    "Return all routers for registration with the app."
```

```python
@patch
def render_entry_point(
    self: StructureDecompWorkflow,
    request,  # FastHTML request object
    sess  # FastHTML session object
):  # FastHTML component
    "Render the workflow entry point for embedding."
```

```python
def _create_data_loaders(
    workflow: StructureDecompWorkflow  # Workflow instance for service access
):  # (load_sources, load_empty) callables
    "Create data loader functions for StepFlow steps."
```

```python
def _validate_always_true(
    state: Dict[str, Any]  # Workflow state dictionary
) -> bool:  # Always True
    "No-op validator for steps that are always valid."
```

```python
def _validate_selection(
    state: Dict[str, Any]  # Workflow state dictionary
) -> bool:  # True if at least one source is selected
    "Validate that sources have been selected."
```

```python
def _create_selection_renderer(
    workflow: StructureDecompWorkflow  # Workflow instance for URL access
):  # Render callable for StepFlow
    "Create render function for the source selection step."
```

```python
def _create_combined_renderer(
    workflow: StructureDecompWorkflow  # Workflow instance with _sa_result set
):  # Render callable for StepFlow
    "Create render function for the segmentation & alignment step."
```

```python
def _create_review_renderer(
    workflow: StructureDecompWorkflow  # Workflow instance for URL and state access
):  # Render callable for StepFlow
    "Create render function for the review step."
```

```python
def _create_review_hook(
    workflow: StructureDecompWorkflow  # Workflow instance for service and state access
):  # on_leave callable
    "Create on_leave hook for the review step (commits to graph)."
```

```python
def _create_verify_renderer(
    workflow: StructureDecompWorkflow  # Workflow instance for URL access
):  # Render callable for StepFlow
    "Create render function for the verify step."
```

```python
def _create_verify_hooks(
    workflow: StructureDecompWorkflow  # Workflow instance for service and state access
):  # (on_enter_verify, on_complete) callables
    "Create lifecycle hooks for the verify step."
```

```python
@patch
def _create_step_flow(
    self: StructureDecompWorkflow
) -> StepFlow:  # Configured StepFlow instance
    "Create and configure the StepFlow instance."
```

```python
@patch
def _create_routers(
    self: StructureDecompWorkflow
) -> List[APIRouter]:  # List of configured APIRouters
    "Create the workflow's API routers."
```

#### Classes

```python
class _SessionStateStoreAdapter:
    def __init__(
        self,
        store: SQLiteWorkflowStateStore  # The pure, framework-agnostic store
    )
    "Adapter bridging StepFlow's sess-based protocol to the pure session_id-based store."
    
    def __init__(
            self,
            store: SQLiteWorkflowStateStore  # The pure, framework-agnostic store
        )
        "Wrap a pure store with session resolution."
    
    def get_current_step(
            self,
            flow_id: str,  # Workflow identifier
            sess: Any  # FastHTML session object
        ) -> Optional[str]:  # Current step ID or None
        "Get current step ID, resolving session from sess."
    
    def set_current_step(
            self,
            flow_id: str,  # Workflow identifier
            sess: Any,  # FastHTML session object
            step_id: str  # Step ID to set as current
        ) -> None
        "Set current step ID, resolving session from sess."
    
    def get_state(
            self,
            flow_id: str,  # Workflow identifier
            sess: Any  # FastHTML session object
        ) -> Dict[str, Any]:  # Workflow state dictionary
        "Get all workflow state, resolving session from sess."
    
    def update_state(
            self,
            flow_id: str,  # Workflow identifier
            sess: Any,  # FastHTML session object
            updates: Dict[str, Any]  # State updates to apply
        ) -> None
        "Update workflow state, resolving session from sess."
    
    def clear_state(
            self,
            flow_id: str,  # Workflow identifier
            sess: Any  # FastHTML session object
        ) -> None
        "Clear all workflow state, resolving session from sess."
```

```python
class StructureDecompWorkflow:
    def __init__(
        self,
        plugin_manager: PluginManager,  # Plugin manager from host application
        config: Optional[StructureDecompWorkflowConfig] = None  # Workflow configuration
    )
    "Self-contained structure decomposition workflow."
    
    def __init__(
            self,
            plugin_manager: PluginManager,  # Plugin manager from host application
            config: Optional[StructureDecompWorkflowConfig] = None  # Workflow configuration
        )
        "Initialize the workflow with injected PluginManager."
    
    def create_and_setup(
            cls,
            app,  # FastHTML application instance
            plugin_manager: PluginManager,  # Plugin manager from host application
            config: Optional[StructureDecompWorkflowConfig] = None  # Workflow configuration
        ) -> "StructureDecompWorkflow":  # Configured and setup workflow instance
        "Create, configure, and setup a workflow in one call."
    
    def plugin_manager(self) -> PluginManager:  # Plugin manager instance
            """Access to plugin manager."""
            return self._plugin_manager
        
        @property
        def job_queue(self) -> JobQueue:  # Job queue instance
        "Access to plugin manager."
    
    def job_queue(self) -> JobQueue:  # Job queue instance
            """Access to host-owned job queue."""
            return self._job_queue
        
        @property
        def source_service(self) -> SourceService:  # Source service instance
        "Access to host-owned job queue."
    
    def source_service(self) -> SourceService:  # Source service instance
            """Access to source service."""
            return self._source_service
        
        @property
        def graph_service(self) -> GraphService:  # Graph service instance
        "Access to source service."
    
    def graph_service(self) -> GraphService:  # Graph service instance
            """Access to graph service."""
            return self._graph_service
        
        @property
        def verify_service(self) -> VerifyService:  # Verify service instance
        "Access to graph service."
    
    def verify_service(self) -> VerifyService:  # Verify service instance
            """Access to verify service."""
            return self._verify_service
        
        @property
        def state_store(self) -> SQLiteWorkflowStateStore:  # State store instance
        "Access to verify service."
    
    def state_store(self) -> SQLiteWorkflowStateStore:  # State store instance
            """Access to state store."""
            return self._state_store
        
        @property
        def routers(self) -> List[APIRouter]:  # List of workflow routers
        "Access to state store."
    
    def routers(self) -> List[APIRouter]:  # List of workflow routers
            """Access to workflow routers (excludes stepflow router)."""
            return self._routers
        
        @property
        def stepflow_router(self) -> APIRouter:  # StepFlow-generated router
        "Access to workflow routers (excludes stepflow router)."
    
    def stepflow_router(self) -> APIRouter:  # StepFlow-generated router
        "StepFlow-generated router."
```
